<a href="https://colab.research.google.com/github/dhunsyam/Advance-Databases/blob/gh-pages/notebooks/Monkeys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import models, layers, optimizers
from keras.layers import Input, Dense
from google.colab import drive
drive.mount('/content/drive')
import os
all_monkeys =[]
train_dir = '/content/drive/MyDrive/training/'
subdir=('n0','n1','n2','n3','n4','n5','n6','n7','n8','n9')
for sub in subdir:
  dir = train_dir + sub +'/'
  print("looking in " + dir)
  for image in os.listdir(dir):
    try:
      monkey = tf.keras.utils.load_img ((dir+ image),
                             target_size = (64,64))
      all_monkeys.append(monkey)
    except Exception as e:
      pass
print ('Recovered data format:', type(all_monkeys))
print('Number of monkey images:', len(all_monkeys))
all_monkeys
plt.imshow(all_monkeys[50])


#Preprocessing the Data

#We will convert our list of pixel values into NumPy arrays. After that, we will normalise the pixels to rang 0 -1 by dividing by 255. Then we flatten the four dimensional array into a two dimensional array, since our deep autoencoder is composed of a feed forward neural network that propagates 2D vectors through its layers.

#Make the array
all_monkeys = np.asarray(all_monkeys)
np.asarray(all_monkeys)
print('Shape of array:', all_monkeys.shape)

#Normalise pixel values
all_monkeys = all_monkeys.astype('float32') / 255

#Flatten array

all_monkeys = all_monkeys.reshape((len(all_monkeys),
                                   np.prod(all_monkeys.shape[1:])))
print('Shape after flattened:', all_monkeys.shape)
#Partioning the Data

#We now need to split the data into training and testing. We will do not use labels as the autoencoder's purpose is not to classify. It is rather to build images back up from the sparse latent representation. We will use shlearn's model selection module to make an 80/20 split ratio.

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(all_monkeys,all_monkeys,
                                        test_size=0.2,random_state= 42)
print("x_train: ", x_train.shape)
print("y_train: ",y_train.shape)
print("x_test: ",x_test.shape)
print("y_test: ",y_test.shape)
#Using Functional API to Design Autoencoder

#We will use the functional API to construct the autoencoder. We will define the input dimension for the images as (64 x 64 x3 = 12288) and an encoding dimsension as (256). This means that each image will be compressed by a factor of 48.

#Input Dimension
input_dim = 12288

#Encoding dimension for the latent space
encoding_dim = 256
#Building the Model

#The input layer will accept dimensions corresponding to our 2D vectors of monkey images. Then we define the encoder part of the network using dense layers, with a decreasing number of neurons for subsequent layers, until the latent space is reached. We will simply choose the number of neurons in layers leading to the latent space to decrease by a factor of 2.

#Input layer placeholder
input_layer = Input(shape=(input_dim,))

#Encoding layers funnel the images into lower dimensional representations
encoded = Dense(encoding_dim*4, activation='relu')(input_layer)
encoded = Dense(encoding_dim*2, activation='relu')(encoded)

#Latent space
encoded = Dense(encoding_dim, activation = 'relu')(encoded)

# 'decoded' is the lossy reconstruction of the input
decoded = Dense(encoding_dim*2, activation='relu')(encoded)
decoded = Dense(encoding_dim*4, activation='relu')(decoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

#this model maps an input to its reconstruction
autoencoder = models.Model(input_layer, decoded)

autoencoder.summary()
#Now let's run train the autoencoder

#We will use the Adam optimise and a mean-square-error loss function

autoencoder.compile(optimizer='adam', loss ='mse')
autoencoder.fit(x_train, x_train, epochs= 50, batch_size=20, verbose=1)
#Visualising the Results

#We will take the encoder part of the network to make predictions and then decode

y = autoencoder.predict(x_train)
y.shape
(963, 12288)
plt.figure (figsize = (22,6))
num_imgs = 9
for i in range(num_imgs):
  #display original
  ax = plt.subplot(2, num_imgs, i+1)
  true_img = x_train[i].reshape(64,64,3)
  plt.imshow(true_img,plt.gray())

  #display reconstruction
  ax = plt.subplot(2, num_imgs, i+1+num_imgs)
  reconstructed_img = y[i].reshape(64,64,3)
  plt.imshow(reconstructed_img, plt.gray())
plt.show()

#The autencoder seems to have worked very well. But let's try it with test data rather than the data it trained on.

#Let's plot the figures. We'll change the earlier code for plotting a little so that we can vary the number of images plotted and we'll set the axes to not visible

#Predict using x_test
decoded_imgs =autoencoder.predict(x_test)
n=6
plt.figure(figsize=(22,6))
for i in range(n):
  #display orginal
  ax = plt.subplot(2,n,i+1)
  plt.imshow(x_test[i].reshape(64,64,3))
  plt.gray()

  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  #display reconstruction
  ax=  plt.subplot(2,n,i+1+n)
  plt.imshow(decoded_imgs[i].reshape(64,64,3))
  plt.gray()

  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)
plt.show()

#Some of the images are a bit fuzzy. Perhaps a CNN autoencoder could do better

#Deep Convolutional Autoencoder

#We will import some convolutional, MaxPooling and Upsampling layers, and start building the network. We will alternate the convolutional and pooling layers until we reach the latent space, which is represented by the second MaxPooling 2D layer

from keras.layers import Conv2D, MaxPooling2D, UpSampling2D
#Input Placeholders
input_img = Input(shape=(64,64,3))

#Encoder part
l1 = Conv2D(32, (3,3), activation ='relu', padding ='same')(input_img)
l2 = MaxPooling2D((2,2),padding='same')(l1)
l3 = Conv2D(16, (3,3), activation ='relu', padding ='same')(l2)

#Latent Space, with dimension (None,32,32,16)
encoded = MaxPooling2D((1,1), padding = 'same')(l3)

#Decoder part
l8 = Conv2D(16, (3,3), activation = 'relu', padding ='same')(encoded)
l9 = UpSampling2D((2,2))(l8)
decoded = Conv2D(3, (3,3), activation ='sigmoid', padding = 'same')(l9)

autoencoderCNN = models.Model(input_img, decoded)

autoencoderCNN.summary()
#to check the shape of a layer

decoded.shape
TensorShape([None, 64, 64, 3])
#Compiling and Training the model

#We will compile out network with the same optimiser and loss function that we chose for the deep feed=forwad network.

autoencoderCNN.compile(optimizer='adam', loss='mse')

#x-train is currently in the shape shown below.

x_train.shape
(963, 12288)
#Let's reshape it and give it a variable name.

x_train1=x_train.reshape(963,64,64,3)
#Now we can fit the data and train theCNN autoencoder

autoencoderCNN.fit(x_train1, x_train1, epochs=50, batch_size=20, shuffle=True, verbose=1)
#Note that the loss for the CNN autoencoder. Is the CNN encoder less less or higher than the NN encoder loss?

#Let's now judge for ourselves how the model performs at reconstructing images it has never seen before.

#Testing and Visualising Results

#We define the function below to compare the text images with their reconstruction

def compare_outputs(x_test, decoded_imgs=None, n=10):
  plt.figure(figsize=(22,5))
  for i in range(n):
    ax=plt.subplot(2,n,i+1)
    plt.imshow(x_test[i].reshape(64,64,3))
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
  for i in range(n):
    if decoded_imgs is not None:
      ax = plt.subplot(2, n, i+1+n)
      plt.imshow(decoded_imgs[i].reshape(64,64,3))
      ax.get_xaxis().set_visible(False)
      ax.get_yaxis().set_visible(False)
  plt.show()
#Now let's predict

#First reshape the test data

x_test1=x_test.reshape(241,64,64,3)
#Now predict

decoded_imgs_CNN = autoencoderCNN.predict(x_test1)
print('Upper row: Input image provided')
print('Bottom row: Decoded output generated')
#Next call the function to compare the test images with their reconstruction.

compare_outputs(x_test1, decoded_imgs_CNN)
#the reconstructed images with the CNN autoencoder are quite a bit sharper than those with the feed forward autoencoder.

MessageError: Error: credential propagation was unsuccessful